In [1]:
# ── Cell 0: HuggingFace login ──────────────────────────────────
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
login(token=user_secrets.get_secret("HF_TOKEN1"))
print("HF login: OK")

HF login: OK


In [2]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.9 MB/s eta 0:00:00


In [3]:
# ── Cell 2: Imports ────────────────────────────────────────────
import os, re, time, warnings
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import math, nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
warnings.filterwarnings("ignore")
print("Imports: OK")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Imports: OK


In [4]:
# ── Cell 3: Config ─────────────────────────────────────────────
TRAIN_TXT  = "/kaggle/input/datasets/softhell/med-vqa-2021/data/train/label.txt"
TRAIN_IMGS = "/kaggle/input/datasets/softhell/med-vqa-2021/data/train/images"
VAL_TXT    = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/label.txt"
VAL_IMGS   = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/images"
EVAL_CSV   = "/kaggle/working/pubmedclip_metrics.csv"
OUTPUT_DIR = "/kaggle/working/pubmedclip_mlp"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID   = "pubmed-clip-vit-base-patch32"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

# Training
LEARNING_RATE = 5e-4
NUM_EPOCHS    = 30       # fast per epoch — can afford more
BATCH_SIZE    = 32
DROPOUT       = 0.3         #0.3

print(f"Device : {DEVICE}")
print(f"Output : {OUTPUT_DIR}")

Device : cuda
Output : /kaggle/working/pubmedclip_mlp


In [5]:
# ── Cell 4: Helpers ────────────────────────────────────────────
def normalise(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def get_synonyms(answer):
    raw   = str(answer).strip()
    parts = [p.strip() for p in raw.split(",")]
    if any(len(p.split()) >= 2 for p in parts):
        return [normalise(raw)]
    return [normalise(p) for p in parts if normalise(p)]

def exact_match_any(pred, answer):
    return int(normalise(pred) in get_synonyms(answer))

print("Helpers: OK")

Helpers: OK


In [6]:
# ── Cell 5: Build answer vocabulary from training set ──────────
#
# BiomedCLIP is a classifier — it needs a fixed answer vocabulary.
# We build this from the training labels only (no val leakage).

train_df = pd.read_csv(TRAIN_TXT, sep="|",
                       names=["image_id", "question", "answer"], header=None)

# Collect all unique normalised answers
all_answers = []
for ans in train_df["answer"]:
    syns = get_synonyms(str(ans))
    if syns:
        all_answers.append(syns[0])

# Build vocab — sorted for reproducibility
vocab      = sorted(set(all_answers))
label2idx  = {label: i for i, label in enumerate(vocab)}
idx2label  = {i: label for label, i in label2idx.items()}
NUM_CLASSES = len(vocab)

print(f"Training vocabulary size : {NUM_CLASSES} unique classes")
print(f"Sample classes           : {vocab[:5]}")

Training vocabulary size : 330 unique classes
Sample classes           : ['abdominal abscess', 'abdominal aortic aneurysm', 'abdominal compartment syndrome', 'aberrant right subclavian artery arsa', 'achalasia']


In [7]:
# ── Cell 6: Dataset ────────────────────────────────────────────
class VQAMedCLIPDataset(Dataset):
    def __init__(self, txt_path, img_dir, label2idx, split="train"):
        df = pd.read_csv(txt_path, sep="|",
                         names=["image_id", "question", "answer"], header=None)
        self.records   = []
        self.label2idx = label2idx
        skipped        = 0

        for _, row in df.iterrows():
            img_id   = str(row["image_id"]).strip()
            fname    = img_id if img_id.endswith((".jpg",".png")) else img_id+".jpg"
            path     = os.path.join(img_dir, fname)
            if not os.path.exists(path):
                continue

            syns = get_synonyms(str(row["answer"]))
            if not syns:
                continue

            primary = syns[0]
            # For training: skip answers not in vocab
            # For val: keep all (OOV gets -1, always wrong)
            label_idx = label2idx.get(primary, -1)
            if split == "train" and label_idx == -1:
                skipped += 1
                continue

            self.records.append({
                "img_path":   path,
                "question":   str(row["question"]).strip(),
                "answer":     primary,
                "raw_answer": str(row["answer"]).strip(),
                "label_idx":  label_idx,
            })

        print(f"  [{split}] Loaded: {len(self.records)} | Skipped OOV: {skipped}")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        return {
            "img_path":   r["img_path"],
            "question":   r["question"],
            "answer":     r["answer"],
            "raw_answer": r["raw_answer"],
            "label_idx":  r["label_idx"],
        }

In [8]:
#cell 7
class PubMedCLIPClassifier(nn.Module):
    def __init__(self, clip_model, num_classes, dropout=0.3):
        super().__init__()
        self.clip = clip_model
        embed_dim = clip_model.config.projection_dim
        # self.mlp = nn.Sequential(
        #      nn.Linear(embed_dim * 2, 512),  # Reduced from 1024
        #      nn.LayerNorm(512),
        #      nn.ReLU(),
        #      nn.Dropout(dropout),
        #      nn.Linear(512, 256),           # Reduced from 512
        #      nn.LayerNorm(256),
        #      nn.ReLU(),
        #      nn.Dropout(dropout),
        #      nn.Linear(256, num_classes),
        #   )
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, 1024),
            nn.LayerNorm(1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        img_feat = self.clip.get_image_features(pixel_values=pixel_values).pooler_output   # <-- must be get_image_features, NOT vision_model
        txt_feat = self.clip.get_text_features(input_ids=input_ids,
                                                attention_mask=attention_mask).pooler_output
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
        fused = torch.cat([img_feat, txt_feat], dim=-1)
        return self.mlp(fused)

In [9]:
# ── Cell 8: Load PubMedCLIP with LoRA ────────────────────────────────────
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model

PUBMED_CLIP_ID = "flaviagiammarino/pubmed-clip-vit-base-patch32"
print("Loading PubMedCLIP model + processor …")
clip_model = CLIPModel.from_pretrained(PUBMED_CLIP_ID).to(DEVICE)
processor  = CLIPProcessor.from_pretrained(PUBMED_CLIP_ID)

print("Injecting LoRA adapters into CLIP …")
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.2,
    bias="none",
)
clip_model = get_peft_model(clip_model, lora_config)
clip_model.print_trainable_parameters()

print("\nBuilding classifier …")
model = PubMedCLIPClassifier(clip_model, NUM_CLASSES, DROPOUT).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total_params/1e6:.2f}M")
print(f"Trainable params : {trainable_params/1e6:.2f}M")

Loading PubMedCLIP model + processor …


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: flaviagiammarino/pubmed-clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

... (4 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Injecting LoRA adapters into CLIP …
trainable params: 1,966,080 || all params: 153,243,393 || trainable%: 1.2830

Building classifier …
Total params     : 154.99M
Trainable params : 3.71M


In [10]:
# ── Cell 9: DataLoaders with train-only augmentation (HF processor) ──
from torchvision import transforms as T

IMG_SIZE = 224  # matches PubMedCLIP's expected input size

train_augment = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    T.RandomRotation(degrees=7),
    T.ColorJitter(brightness=0.15, contrast=0.15),
])

def collate_train(batch):
    questions  = [s["question"] for s in batch]
    img_paths  = [s["img_path"] for s in batch]
    label_idxs = torch.tensor([s["label_idx"] for s in batch], dtype=torch.long)
    answers     = [s["answer"]     for s in batch]
    raw_answers = [s["raw_answer"] for s in batch]

    images = [train_augment(Image.open(p).convert("RGB")) for p in img_paths]

    enc = processor(text=questions, images=images, return_tensors="pt",
                     padding=True, truncation=True)

    return {"pixel_values": enc["pixel_values"], "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"], "label_idx": label_idxs,
            "answer": answers, "raw_answer": raw_answers}

def collate_val(batch):
    questions  = [s["question"] for s in batch]
    img_paths  = [s["img_path"] for s in batch]
    label_idxs = torch.tensor([s["label_idx"] for s in batch], dtype=torch.long)
    answers     = [s["answer"]     for s in batch]
    raw_answers = [s["raw_answer"] for s in batch]

    images = [Image.open(p).convert("RGB") for p in img_paths]  # no augmentation

    enc = processor(text=questions, images=images, return_tensors="pt",
                     padding=True, truncation=True)

    return {"pixel_values": enc["pixel_values"], "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"], "label_idx": label_idxs,
            "answer": answers, "raw_answer": raw_answers}

print("\nBuilding datasets …")
train_dataset = VQAMedCLIPDataset(TRAIN_TXT, TRAIN_IMGS, label2idx, "train")
val_dataset   = VQAMedCLIPDataset(VAL_TXT,   VAL_IMGS,   label2idx, "val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_train,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_val,
                          num_workers=2, pin_memory=True)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")


Building datasets …
  [train] Loaded: 4500 | Skipped OOV: 0
  [val] Loaded: 500 | Skipped OOV: 0
Train batches : 141
Val batches   : 16


In [11]:
# ── Cell 10: Optimizer + scheduler (LoRA Version, regularized + early stopping) ──
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if "mlp" in n and p.requires_grad], "weight_decay": 5e-3}, #change 5e-3
    {"params": [p for n, p in model.named_parameters() if "mlp" not in n and p.requires_grad], "weight_decay": 1e-3},
], lr=LEARNING_RATE)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS)

criterion = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.1)
best_val_acc = 0.0
best_ckpt    = os.path.join(OUTPUT_DIR, "best_model.pt")

patience = 6
epochs_no_improve = 0

print(f"Training LoRA + MLP Head for {NUM_EPOCHS} Epochs...")

for epoch in range(1, NUM_EPOCHS + 1):
    
    # ── Train ──────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]"):
        pixel_values   = batch["pixel_values"].to(DEVICE)
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        lbl            = batch["label_idx"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(pixel_values, input_ids, attention_mask)
        loss   = criterion(logits, lbl)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()

        train_loss    += loss.item()
        preds          = logits.argmax(dim=-1)
        valid          = lbl != -1
        train_correct += (preds[valid] == lbl[valid]).sum().item()
        train_total   += valid.sum().item()

    scheduler.step()
    train_acc = train_correct / max(train_total, 1) * 100

    # ── Validation ─────────────────────────────────────────────
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            pixel_values   = batch["pixel_values"].to(DEVICE)
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            lbl            = batch["label_idx"].to(DEVICE)
            
            logits = model(pixel_values, input_ids, attention_mask)
            preds  = logits.argmax(dim=-1)
            valid  = lbl != -1
            val_correct += (preds[valid] == lbl[valid]).sum().item()
            val_total   += valid.sum().item()

    val_acc = val_correct / max(val_total, 1) * 100
    print(f"  Epoch {epoch:2d} — loss: {train_loss/len(train_loader):.4f} | "
          f"train: {train_acc:.2f}% | val: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_ckpt)
        print(f"  ✓ Best → {best_ckpt}  ({best_val_acc:.2f}%)")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\n⏹ Early stopping at epoch {epoch} "
                  f"(no val improvement for {patience} epochs)")
            break

print(f"\nTraining complete. Best val acc: {best_val_acc:.2f}%")

Training LoRA + MLP Head for 30 Epochs...


Epoch 1/30 [train]: 100%|██████████| 141/141 [00:40<00:00,  3.48it/s]


  Epoch  1 — loss: 5.3965 | train: 5.27% | val: 9.60%
  ✓ Best → /kaggle/working/pubmedclip_mlp/best_model.pt  (9.60%)


Epoch 2/30 [train]: 100%|██████████| 141/141 [00:38<00:00,  3.68it/s]


  Epoch  2 — loss: 4.4382 | train: 12.27% | val: 11.80%
  ✓ Best → /kaggle/working/pubmedclip_mlp/best_model.pt  (11.80%)


Epoch 3/30 [train]: 100%|██████████| 141/141 [00:37<00:00,  3.75it/s]


... (51 outputs left out)

Epoch 29/30 [train]: 100%|██████████| 141/141 [00:38<00:00,  3.67it/s]


  Epoch 29 — loss: 1.0783 | train: 99.73% | val: 56.60%


Epoch 30/30 [train]: 100%|██████████| 141/141 [00:38<00:00,  3.66it/s]


  Epoch 30 — loss: 1.0735 | train: 99.80% | val: 57.00%

Training complete. Best val acc: 57.20%


In [12]:
# # ── Cell 11: Full evaluation with Semantic Match & BLEU ────
# import torch.nn.functional as F

# print("Loading best checkpoint …")
# model.load_state_dict(torch.load(best_ckpt))
# model.eval()

# vocab_enc = processor(text=vocab, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
# with torch.no_grad():
#     vocab_txt_feats = clip_model.get_text_features(
#         input_ids=vocab_enc["input_ids"], attention_mask=vocab_enc["attention_mask"]).pooler_output
#     vocab_txt_feats = vocab_txt_feats / vocab_txt_feats.norm(dim=-1, keepdim=True)

# val_df   = pd.read_csv(VAL_TXT, sep="|",
#                        names=["image_id","question","answer"], header=None)
# smoothie = SmoothingFunction().method1

# results      = []
# total        = 0
# exact_sum    = 0
# semantic_sum = 0
# bleu1_sum    = 0.0
# bleu2_sum    = 0.0
# bleu3_sum    = 0.0
# bleu4_sum    = 0.0
# bleu_cum_sum = 0.0

# # Tracker to help you calibrate your threshold
# wrong_sim_scores = []

# def compute_bleu_ngrams(pred_tokens, gt_tokens, smoothfn):
#     refs = [gt_tokens]
#     hyp  = pred_tokens
#     b1   = sentence_bleu(refs, hyp, weights=(1,0,0,0), smoothing_function=smoothfn)
#     b2   = sentence_bleu(refs, hyp, weights=(0,1,0,0), smoothing_function=smoothfn)
#     b3   = sentence_bleu(refs, hyp, weights=(0,0,1,0), smoothing_function=smoothfn)
#     b4   = sentence_bleu(refs, hyp, weights=(0,0,0,1), smoothing_function=smoothfn)
#     cum  = math.exp(
#         (math.log(b1+1e-10) + math.log(b2+1e-10) +
#          math.log(b3+1e-10) + math.log(b4+1e-10)) / 4)
#     return b1, b2, b3, b4, cum

# # INCREASED THRESHOLD to 0.98 to account for CLIP text anisotropy
# def compute_semantic_match(pred_str, gt_str, threshold=0.99):    
#     """Uses PubMedCLIP text encoder to check if answers mean the same thing."""
#     if pred_str == gt_str:
#         return 1.0, 1
        
#     inputs = processor(text=[pred_str, gt_str], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
#     with torch.no_grad():
#         feats = clip_model.get_text_features(**inputs)
        
#         # Safely extract the tensor from the LoRA/PEFT wrapper object
#         if hasattr(feats, 'pooler_output'):
#             feats = feats.pooler_output
#         elif hasattr(feats, 'text_embeds'):
#             feats = feats.text_embeds
            
#         # Now we can safely normalize
#         feats = F.normalize(feats, p=2, dim=-1)
        
#     # Cosine similarity between pred (idx 0) and gt (idx 1)
#     sim = (feats[0] @ feats[1].T).item()
#     return sim, int(sim >= threshold)


# print(f"Evaluating on {len(val_df)} samples …\n")

# for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
#     img_id     = str(row["image_id"]).strip()
#     question   = str(row["question"]).strip()
#     raw_answer = str(row["answer"]).strip()
#     primary_gt = get_synonyms(raw_answer)[0]

#     fname    = img_id if img_id.endswith((".jpg",".png")) else img_id+".jpg"
#     img_path = os.path.join(VAL_IMGS, fname)
#     if not os.path.exists(img_path):
#         continue

#     image = Image.open(img_path).convert("RGB")
#     enc = processor(text=[question], images=[image], return_tensors="pt",
#                      padding=True, truncation=True).to(DEVICE)

#     with torch.no_grad():
#         logits = model(enc["pixel_values"], enc["input_ids"], enc["attention_mask"])
#         pred_idx = logits.argmax(dim=-1).item()

#     pred = idx2label.get(pred_idx, "unknown")

#     # Metrics
#     em               = exact_match_any(pred, raw_answer)
#     sim_score, sem_m = compute_semantic_match(pred, primary_gt)
    
#     # If it's an exact match, guarantee semantic match is also 1
#     if em:
#         sem_m = 1
#         sim_score = max(sim_score, 1.0)
#     else:
#         # Track similarities of WRONG answers for calibration
#         wrong_sim_scores.append(sim_score)
        
#     b1,b2,b3,b4,bcum = compute_bleu_ngrams(pred.split(), primary_gt.split(), smoothie)

#     total        += 1
#     exact_sum    += em
#     semantic_sum += sem_m
#     bleu1_sum    += b1
#     bleu2_sum    += b2
#     bleu3_sum    += b3
#     bleu4_sum    += b4
#     bleu_cum_sum += bcum

#     # Included BLEU 1-4 columns into the saved output data
#     results.append({
#         "Image ID":        img_id,
#         "Question":        question,
#         "Ground Truth":    raw_answer,
#         "Prediction":      pred,
#         "Exact Match":     em,
#         "Semantic Match":  sem_m,
#         "Sim Score":       round(sim_score, 4),
#         "BLEU-1":          round(b1,   4),
#         "BLEU-2":          round(b2,   4),
#         "BLEU-3":          round(b3,   4),
#         "BLEU-4":          round(b4,   4),
#         "Cumulative BLEU": round(bcum, 4),
#     })

# results_df = pd.DataFrame(results)
# results_df.to_csv(EVAL_CSV, index=False)

# avg_wrong_sim = sum(wrong_sim_scores) / len(wrong_sim_scores) if wrong_sim_scores else 0

# print(f"\nResults → {EVAL_CSV}")
# print("\n" + "=" * 57)
# print("   PUBMEDCLIP + MLP — VQA-MED 2021 RESULTS")
# print("=" * 57)
# print(f"  Total             : {total}")
# print(f"  Exact accuracy    : {exact_sum    / total * 100:.2f}%")
# print(f"  Semantic accuracy : {semantic_sum / total * 100:.2f}%")
# print(f"  BLEU-1            : {bleu1_sum    / total * 100:.2f}")
# print(f"  BLEU-2            : {bleu2_sum    / total * 100:.2f}")
# print(f"  BLEU-3            : {bleu3_sum    / total * 100:.2f}")
# print(f"  BLEU-4            : {bleu4_sum    / total * 100:.2f}")
# print(f"  Cumulative BLEU   : {bleu_cum_sum / total * 100:.2f}")
# print("-" * 57)
# print(f"  Avg Sim of Wrong  : {avg_wrong_sim:.4f}  <-- Adjust threshold above this!")
# print("=" * 57)


# ── Cell 11: Full evaluation with Unified Semantic Match & BLEU ────
import re
import numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

print("Loading best checkpoint …")
model.load_state_dict(torch.load(best_ckpt))
model.eval()

# (Keeping your original vocab encoding setup intact just in case)
vocab_enc = processor(text=vocab, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
with torch.no_grad():
    vocab_txt_feats = clip_model.get_text_features(
        input_ids=vocab_enc["input_ids"], attention_mask=vocab_enc["attention_mask"]).pooler_output
    vocab_txt_feats = vocab_txt_feats / vocab_txt_feats.norm(dim=-1, keepdim=True)

# 1. Centralized Independent Evaluator (Run on CPU to safeguard VRAM)
print("Initializing Centralized Evaluator (Standalone BiomedBERT)...")
BIOMEDBERT_ID = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
BIO_TOKENIZER = AutoTokenizer.from_pretrained(BIOMEDBERT_ID)
BIO_EMBEDDER  = AutoModel.from_pretrained(BIOMEDBERT_ID).to("cpu")
BIO_EMBEDDER.eval()
print(f"✓ Loaded {BIOMEDBERT_ID}\n")

def compute_unified_semantic_match(pred_str, gt_str, threshold=0.85):
    """Unified evaluation function to eliminate contrastive anisotropy bias."""
    if not pred_str or not gt_str:
        return 0.0, 0
        
    p_clean = re.sub(r"[^\w\s]", " ", str(pred_str).lower()).strip()
    p_clean = re.sub(r"\s+", " ", p_clean)
    g_clean = re.sub(r"[^\w\s]", " ", str(gt_str).lower()).strip()
    g_clean = re.sub(r"\s+", " ", g_clean)
    
    if p_clean == g_clean:
        return 1.0, 1

    inputs = BIO_TOKENIZER([p_clean, g_clean], padding=True, truncation=True, return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        outputs = BIO_EMBEDDER(**inputs)
        attention_mask = inputs["attention_mask"]
        token_embeddings = outputs[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        embeddings = F.normalize(embeddings, p=2, dim=-1)
        
    sim = (embeddings[0] @ embeddings[1].T).item()
    return sim, int(sim >= threshold)

def print_calibration_report(wrong_sim_scores, threshold):
    if not wrong_sim_scores:
        print("No wrong-answer similarity scores collected.")
        return
    arr = np.array(wrong_sim_scores)
    print("\n" + "=" * 57)
    print("  SEMANTIC-MATCH THRESHOLD CALIBRATION CHECK")
    print("=" * 57)
    print(f"  Threshold in use : {threshold:.4f}")
    print(f"  Wrong-sim min    : {arr.min():.4f}")
    print(f"  Wrong-sim mean   : {arr.mean():.4f}")
    print(f"  Wrong-sim p90    : {np.percentile(arr, 90):.4f}")
    print(f"  Wrong-sim p95    : {np.percentile(arr, 95):.4f}")
    print(f"  Wrong-sim max    : {arr.max():.4f}")
    n_leaking = int((arr >= threshold).sum())
    print(f"  Wrong answers passing threshold: {n_leaking}/{len(arr)} ({n_leaking/len(arr)*100:.1f}%)")
    if n_leaking / len(arr) > 0.02:
        print("  ⚠ More than 2% of wrong answers are passing the semantic-match "
              "threshold. Consider raising the threshold.")
    print("=" * 57)

def compute_bleu_ngrams(pred_tokens, gt_tokens, smoothfn):
    refs = [gt_tokens]
    hyp  = pred_tokens
    b1   = sentence_bleu(refs, hyp, weights=(1,0,0,0), smoothing_function=smoothfn)
    b2   = sentence_bleu(refs, hyp, weights=(0,1,0,0), smoothing_function=smoothfn)
    b3   = sentence_bleu(refs, hyp, weights=(0,0,1,0), smoothing_function=smoothfn)
    b4   = sentence_bleu(refs, hyp, weights=(0,0,0,1), smoothing_function=smoothfn)
    cum  = math.exp(
        (math.log(b1+1e-10) + math.log(b2+1e-10) +
         math.log(b3+1e-10) + math.log(b4+1e-10)) / 4)
    return b1, b2, b3, b4, cum

# 2. Setup Evaluation Data
val_df   = pd.read_csv(VAL_TXT, sep="|", names=["image_id","question","answer"], header=None)
smoothie = SmoothingFunction().method1
results      = []
total        = 0
exact_sum    = 0
semantic_sum = 0
bleu1_sum    = 0.0
bleu2_sum    = 0.0
bleu3_sum    = 0.0
bleu4_sum    = 0.0
bleu_cum_sum = 0.0
wrong_sim_scores = []

# --- YOU CAN TWEAK THIS THRESHOLD BASED ON THE CALIBRATION REPORT ---
SEMANTIC_THRESHOLD = 0.96

print(f"Evaluating on {len(val_df)} samples …\n")

# 3. Main Inference Loop
for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    img_id     = str(row["image_id"]).strip()
    question   = str(row["question"]).strip()
    raw_answer = str(row["answer"]).strip()
    primary_gt = get_synonyms(raw_answer)[0]

    fname    = img_id if img_id.endswith((".jpg",".png")) else img_id+".jpg"
    img_path = os.path.join(VAL_IMGS, fname)
    if not os.path.exists(img_path):
        continue

    image = Image.open(img_path).convert("RGB")
    enc = processor(text=[question], images=[image], return_tensors="pt",
                     padding=True, truncation=True).to(DEVICE)

    with torch.no_grad():
        logits = model(enc["pixel_values"], enc["input_ids"], enc["attention_mask"])
        pred_idx = logits.argmax(dim=-1).item()

    pred = idx2label.get(pred_idx, "unknown")

    # Metrics
    em = exact_match_any(pred, raw_answer)
    sim_score, sem_m = compute_unified_semantic_match(pred, primary_gt, threshold=SEMANTIC_THRESHOLD)
    
    if em:
        sem_m = 1
        sim_score = max(sim_score, 1.0)
    else:
        wrong_sim_scores.append(sim_score)
        
    b1,b2,b3,b4,bcum = compute_bleu_ngrams(pred.split(), primary_gt.split(), smoothie)

    total        += 1
    exact_sum    += em
    semantic_sum += sem_m
    bleu1_sum    += b1
    bleu2_sum    += b2
    bleu3_sum    += b3
    bleu4_sum    += b4
    bleu_cum_sum += bcum

    results.append({
        "Image ID":        img_id,
        "Question":        question,
        "Ground Truth":    raw_answer,
        "Prediction":      pred,
        "Exact Match":     em,
        "Semantic Match":  sem_m,
        "Sim Score":       round(sim_score, 4),
        "BLEU-1":          round(b1,   4),
        "BLEU-2":          round(b2,   4),
        "BLEU-3":          round(b3,   4),
        "BLEU-4":          round(b4,   4),
        "Cumulative BLEU": round(bcum, 4),
    })

# 4. Save and Report
results_df = pd.DataFrame(results)
results_df.to_csv(EVAL_CSV, index=False)
avg_wrong_sim = sum(wrong_sim_scores) / len(wrong_sim_scores) if wrong_sim_scores else 0

print(f"\nResults → {EVAL_CSV}")
print("\n" + "=" * 57)
print("   PUBMEDCLIP + MLP — VQA-MED 2021 RESULTS")
print("=" * 57)
print(f"  Total             : {total}")
print(f"  Exact accuracy    : {exact_sum    / total * 100:.2f}%")
print(f"  Semantic accuracy : {semantic_sum / total * 100:.2f}%")
print(f"  BLEU-1            : {bleu1_sum    / total * 100:.2f}")
print(f"  BLEU-2            : {bleu2_sum    / total * 100:.2f}")
print(f"  BLEU-3            : {bleu3_sum    / total * 100:.2f}")
print(f"  BLEU-4            : {bleu4_sum    / total * 100:.2f}")
print(f"  Cumulative BLEU   : {bleu_cum_sum / total * 100:.2f}")
print("-" * 57)
print(f"  Avg Sim of Wrong  : {avg_wrong_sim:.4f}")
print("=" * 57)

# Print the calibration report at the very end
print_calibration_report(wrong_sim_scores, threshold=SEMANTIC_THRESHOLD)

Loading best checkpoint …
Initializing Centralized Evaluator (Standalone BiomedBERT)...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

✓ Loaded microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract

Evaluating on 500 samples …



100%|██████████| 500/500 [00:33<00:00, 15.00it/s]


Results → /kaggle/working/pubmedclip_metrics.csv

   PUBMEDCLIP + MLP — VQA-MED 2021 RESULTS
  Total             : 500
  Exact accuracy    : 57.20%
  Semantic accuracy : 60.20%
  BLEU-1            : 58.44
  BLEU-2            : 46.16
  BLEU-3            : 29.71
  BLEU-4            : 16.38
  Cumulative BLEU   : 28.26
---------------------------------------------------------
  Avg Sim of Wrong  : 0.9185

  SEMANTIC-MATCH THRESHOLD CALIBRATION CHECK
  Threshold in use : 0.9600
  Wrong-sim min    : 0.8389
  Wrong-sim mean   : 0.9185
  Wrong-sim p90    : 0.9533
  Wrong-sim p95    : 0.9658
  Wrong-sim max    : 0.9919
  Wrong answers passing threshold: 15/214 (7.0%)
  ⚠ More than 2% of wrong answers are passing the semantic-match threshold. Consider raising the threshold.


In [13]:
# ── Cell 12: Sample predictions ────────────────────────────────
print("\n── 20 sample predictions ──")
sample = results_df.sample(min(20, len(results_df)), random_state=42)
for _, r in sample.iterrows():
    mark = "✓" if r["Exact Match"] else "✗"
    print(f"  {mark}  GT: {str(r['Ground Truth']):<40}  Pred: {r['Prediction']}")


── 20 sample predictions ──
  ✓  GT: lymphangitic carcinomatosis               Pred: lymphangitic carcinomatosis
  ✓  GT: rheumatoid arthritis                      Pred: rheumatoid arthritis
  ✗  GT: odontoid fracture                         Pred: c spine fracture
  ✗  GT: intraosseous lipoma                       Pred: osteomyelitis
  ✗  GT: pulmonary embolism                        Pred: pulmonary thromboembolism
  ✓  GT: motor vehicle accidents and instrument failure  Pred: motor vehicle accidents and instrument failure
  ✗  GT: round atelectasis                         Pred: lymphangitic carcinomatosis
  ✓  GT: classic bosniak class iii complex renal cyst  Pred: classic bosniak class iii complex renal cyst
  ✗  GT: congenital cystic adenomatoid malformation  Pred: pulmonary sequestration
  ✓  GT: granular cell tumor                       Pred: granular cell tumor
  ✓  GT: cavernous hemangioma                      Pred: cavernous hemangioma
  ✓  GT: liver hemangioma                

In [14]:
# ── PubMedCLIP XAI Occlusion Pipeline (mirrors BiomedCLIP pipeline exactly) ──
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

print("Initializing XAI Occlusion Pipeline for PubMedCLIP...")

def compute_pubmedclip_occlusion(model, pixel_values, input_ids, attention_mask, target_idx,
                                  patch_size=32, stride=16, batch_size=64):
    """
    Sliding-window occlusion sensitivity for the PubMedCLIP + MLP classifier.
    Same windowing/batching strategy as the BiomedCLIP version, adapted to
    PubMedCLIP's 3-argument forward signature: model(pixel_values, input_ids,
    attention_mask) instead of model(image_tensor, text_tokens).
    """
    model.eval()
    _, channels, height, width = pixel_values.shape

    with torch.no_grad():
        base_logits = model(pixel_values, input_ids, attention_mask)
        base_prob = F.softmax(base_logits, dim=-1)[0, target_idx].item()

    y_coords = list(range(0, height - patch_size + 1, stride))
    x_coords = list(range(0, width - patch_size + 1, stride))
    heatmap_grid = np.zeros((len(y_coords), len(x_coords)))

    batch_images = []
    batch_indices = []

    for r_idx, y in enumerate(y_coords):
        for c_idx, x in enumerate(x_coords):
            perturbed_img = pixel_values.clone()
            perturbed_img[0, :, y:y+patch_size, x:x+patch_size] = 0.0  # ~gray under CLIP norm

            batch_images.append(perturbed_img)
            batch_indices.append((r_idx, c_idx))

            if len(batch_images) == batch_size or (y == y_coords[-1] and x == x_coords[-1]):
                b_images = torch.cat(batch_images, dim=0).to(DEVICE)
                b_input_ids      = input_ids.repeat(len(b_images), 1).to(DEVICE)
                b_attention_mask = attention_mask.repeat(len(b_images), 1).to(DEVICE)

                with torch.no_grad():
                    logits = model(b_images, b_input_ids, b_attention_mask)
                    probs = F.softmax(logits, dim=-1)[:, target_idx].cpu().numpy()

                for (r, c), prob in zip(batch_indices, probs):
                    heatmap_grid[r, c] = max(0.0, base_prob - prob)

                batch_images = []
                batch_indices = []

    return heatmap_grid, base_prob


# ── Hallucination Analysis ─────────────────────────────────────────────────
# "Grounded" = occluding the most important region drops confidence by at
# least this much. Identical threshold and rule to the BiomedCLIP script.
VISUAL_RELIANCE_THRESHOLD = 0.10

hallucination_results = []
grounded_correct = 0
ungrounded_correct = 0
hallucinated_wrong = 0

val_df = pd.read_csv(VAL_TXT, sep="|", names=["image_id", "question", "answer"], header=None)

print(f"Running XAI on validation set (n={len(val_df)})...")

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    img_id     = str(row["image_id"]).strip()
    question   = str(row["question"]).strip()
    raw_answer = str(row["answer"]).strip()
    primary_gt = get_synonyms(raw_answer)[0]

    fname    = img_id if img_id.endswith((".jpg", ".png")) else img_id + ".jpg"
    img_path = os.path.join(VAL_IMGS, fname)
    if not os.path.exists(img_path):
        continue

    image = Image.open(img_path).convert("RGB")
    enc = processor(text=[question], images=[image], return_tensors="pt",
                     padding=True, truncation=True).to(DEVICE)

    # Standard prediction
    with torch.no_grad():
        logits = model(enc["pixel_values"], enc["input_ids"], enc["attention_mask"])
        pred_idx = logits.argmax(dim=-1).item()

    pred_str = idx2label.get(pred_idx, "unknown")

    # 1. Semantic match (unified BiomedBERT evaluator, same threshold as Cell 11)
    _, sem_match = compute_unified_semantic_match(pred_str, primary_gt, threshold=SEMANTIC_THRESHOLD)
    if exact_match_any(pred_str, raw_answer):
        sem_match = 1  # guarantee exact match is never overridden, same rule as Cell 11

    # 2. Occlusion on the PREDICTED class
    heatmap, base_conf = compute_pubmedclip_occlusion(
        model, enc["pixel_values"], enc["input_ids"], enc["attention_mask"], pred_idx)
    max_drop = heatmap.max()

    # 3. Categorize
    if sem_match == 1:
        if max_drop >= VISUAL_RELIANCE_THRESHOLD:
            category = "Grounded Correct"
            grounded_correct += 1
        else:
            category = "Ungrounded Correct"
            ungrounded_correct += 1
    else:
        category = "Hallucinated Wrong"
        hallucinated_wrong += 1

    hallucination_results.append({
        "Image ID": fname,
        "Question": question,
        "Ground Truth": raw_answer,
        "Prediction": pred_str,
        "Base Confidence": round(base_conf, 4),
        "Max Conf Drop": round(max_drop, 4),
        "Taxonomy Category": category,
    })

# Save and Output
xai_df = pd.DataFrame(hallucination_results)
xai_df.to_csv(os.path.join(OUTPUT_DIR, "pubmedclip_xai_hallucinations.csv"), index=False)

print("\n" + "=" * 57)
print(f" UNIFIED HALLUCINATION SUMMARY (PubMedCLIP, n={len(xai_df)})")
print("=" * 57)
n = len(xai_df)
print(f"  Grounded Correct   : {grounded_correct}/{n} ({grounded_correct/n*100:.1f}%)")
print(f"  Ungrounded Correct : {ungrounded_correct}/{n} ({ungrounded_correct/n*100:.1f}%)")
print(f"  Hallucinated Wrong : {hallucinated_wrong}/{n} ({hallucinated_wrong/n*100:.1f}%)")
print("=" * 57)

Initializing XAI Occlusion Pipeline for PubMedCLIP...
Running XAI on validation set (n=500)...


100%|██████████| 500/500 [06:06<00:00,  1.37it/s]


 UNIFIED HALLUCINATION SUMMARY (PubMedCLIP, n=500)
  Grounded Correct   : 286/500 (57.2%)
  Ungrounded Correct : 15/500 (3.0%)
  Hallucinated Wrong : 199/500 (39.8%)


In [15]:
# ── PubMedCLIP qualitative XAI verification panel (mirrors BiomedCLIP panel) ──
import cv2
import matplotlib.pyplot as plt

print("Generating qualitative XAI verification panel...")

# 1. Select one distinct representative from each category — middle of the
#    stack per category, same convention as the BiomedCLIP panel.
categories = ["Grounded Correct", "Ungrounded Correct", "Hallucinated Wrong"]
selected_examples = {}

for cat in categories:
    matches = xai_df[xai_df["Taxonomy Category"] == cat]
    if not matches.empty:
        selected_examples[cat] = matches.iloc[len(matches) // 2]

# 2. Setup Plot Grid (3 rows x 2 columns: Original vs. XAI Heatmap)
fig, axes = plt.subplots(3, 2, figsize=(12, 16))
fig.suptitle("PubMedCLIP Hallucination Taxonomy & Visual Grounding Maps",
             fontsize=16, fontweight="bold", y=0.96)

for idx, cat in enumerate(categories):
    if cat not in selected_examples:
        print(f"Skipping visualization for {cat}: No examples found in dataset.")
        continue

    ex = selected_examples[cat]
    img_name = ex["Image ID"]
    img_path = os.path.join(VAL_IMGS, img_name)

    orig_img = Image.open(img_path).convert("RGB")
    img_np = np.array(orig_img)
    h, w, _ = img_np.shape

    # Recompute the specific heatmap at higher resolution for smooth visuals
    question_text = ex["Question"]
    enc = processor(text=[question_text], images=[orig_img], return_tensors="pt",
                     padding=True, truncation=True).to(DEVICE)

    with torch.no_grad():
        logits = model(enc["pixel_values"], enc["input_ids"], enc["attention_mask"])
        pred_idx = logits.argmax(dim=-1).item()

    heatmap, _ = compute_pubmedclip_occlusion(
        model, enc["pixel_values"], enc["input_ids"], enc["attention_mask"], pred_idx,
        patch_size=24, stride=8, batch_size=128
    )

    if heatmap.max() > 0:
        heatmap_norm = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    else:
        heatmap_norm = heatmap

    heatmap_resized = cv2.resize(heatmap_norm, (w, h), interpolation=cv2.INTER_CUBIC)
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(img_np, 0.6, heatmap_color, 0.4, 0)

    # --- Column 1: Original Image + Clinical Metadata ---
    axes[idx, 0].imshow(orig_img)
    axes[idx, 0].set_title(f"Category: {cat}", fontsize=13, fontweight="bold",
                            color="darkred" if "Wrong" in cat else "darkgreen")
    axes[idx, 0].axis("off")

    metadata_text = (
        f"Q: {question_text}\n"
        f"GT: {ex['Ground Truth']}\n"
        f"Pred: {ex['Prediction']}\n"
        f"Confidence Drop: {ex['Max Conf Drop']:.4f}"
    )
    axes[idx, 0].text(10, h - 20, metadata_text, color="black", fontsize=10,
                       weight="bold", bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))

    # --- Column 2: XAI Attention Map Overlay ---
    axes[idx, 1].imshow(overlay)
    axes[idx, 1].set_title(f"Occlusion Sensitivity Map (Target: {ex['Prediction']})",
                            fontsize=11, fontstyle="italic")
    axes[idx, 1].axis("off")

# 3. Clean layout and write to output folder
plt.tight_layout(rect=[0, 0.03, 1, 0.93])
panel_path = os.path.join(OUTPUT_DIR, "pubmedclip_taxonomy_panel.png")
plt.savefig(panel_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"✓ Saved category comparison panel → {panel_path}")

Generating qualitative XAI verification panel...
✓ Saved category comparison panel → /kaggle/working/pubmedclip_mlp/pubmedclip_taxonomy_panel.png
